# ZIP Export Details — EDS Test Bundles

## Overview

This section performs the **final assembly and export** of EDS test data bundles.  
All selected files are staged locally and written to a compressed ZIP archive for **download, review, and external testing**.

The export is intentionally **small, deterministic, and reproducible**, suitable for sharing with developers, reviewers, or downstream tools.

---

## Export Contents

Each ZIP bundle may include:

- **Derived compatibility outputs**
  - Primary raster files (`.img`)
  - All associated ENVI sidecars (`.hdr`, `.aux.xml`, `.ovr`, `.prj`)
- **Matched original input datasets**
  - Surface Reflectance (SR) or Fractional Cover (FC) rasters
  - Automatically resolved based on scene and acquisition date

Only files required to correctly open and interpret the rasters are included.

---

## File Staging and Structure

Before zipping, all files are staged under a temporary export directory to ensure:

- Clean, controlled file selection
- No modification of original EDS or EASI data
- Consistent directory layout across runs

Inside the ZIP archive:

- Directory structure is preserved **relative to the tile root**
- Paths remain consistent with EDS conventions (e.g. `p089r078/...`)

---

## Export Location

All ZIP files are written to:


Existing ZIP files with the same name are **replaced** to allow safe re-runs.

---

## Compression Strategy

ZIP archives are created using:

- `ZIP_DEFLATED` compression
- Moderate compression level (`compresslevel=6`)

This provides a balance between:
- File size reduction
- Fast creation and extraction for large raster datasets

---

## Intended Use

These ZIP bundles are intended for:

- Local raster inspection (QGIS, ArcGIS Pro, GDAL)
- Verification of masking and NoData behaviour
- Targeted debugging of EDS processing logic
- Sharing minimal, self-contained test cases

They are **not intended** to replace full EDS data exports or operational deliveries.


In [51]:
from pathlib import Path
timeseries_data = "ndvi"
base_dir_ = Path("/home/jovyan/work-easi-eds/data/compat/files")

base_dir = base_dir_ / timeseries_data
#base_dir = Path("/home/jovyan/data/compat/files")
assert base_dir.exists(), f"Missing base directory: {base_dir}"

tile_dirs = sorted([p for p in base_dir.iterdir() if p.is_dir()])

print(f"Found {len(tile_dirs)} processed tiles:\n")

for t in tile_dirs:
    n_files = sum(1 for _ in t.rglob("*") if _.is_file())
    print(f"{t.name:<12}  files: {n_files}")

print("\nTile directories:")
for t in tile_dirs:
    print(" ", t)


Found 4 processed tiles:

p089r078      files: 473
p089r084      files: 380
p100r082      files: 497
p104r074      files: 578

Tile directories:
  /home/jovyan/work-easi-eds/data/compat/files/ndvi/p089r078
  /home/jovyan/work-easi-eds/data/compat/files/ndvi/p089r084
  /home/jovyan/work-easi-eds/data/compat/files/ndvi/p100r082
  /home/jovyan/work-easi-eds/data/compat/files/ndvi/p104r074


In [52]:
from pathlib import Path

tile = "p104r074"
tile_dir = base_dir / tile
print("Tile dir:", tile_dir)
print("Exists:", tile_dir.exists())

files = sorted([p for p in tile_dir.rglob("*") if p.is_file()])
print("File count:", len(files))

for p in files[:50]:
    print(p.relative_to(tile_dir))


Tile dir: /home/jovyan/work-easi-eds/data/compat/files/ndvi/p104r074
Exists: True
File count: 578
eds_master_results_104_074_ndvi_d20230503_20231026.json
lztmna_p104r074_eall_dw1mz.hdr
lztmna_p104r074_eall_dw1mz.img
lztmna_p104r074_eall_dw1mz.img.aux.xml
lztmre_p104r074_20151223_dc4ndvi.hdr
lztmre_p104r074_20151223_dc4ndvi.img
lztmre_p104r074_20151223_dc4ndvi.img.aux.xml
lztmre_p104r074_20160209_dc4ndvi.hdr
lztmre_p104r074_20160209_dc4ndvi.img
lztmre_p104r074_20160209_dc4ndvi.img.aux.xml
lztmre_p104r074_20160328_dc4ndvi.hdr
lztmre_p104r074_20160328_dc4ndvi.img
lztmre_p104r074_20160328_dc4ndvi.img.aux.xml
lztmre_p104r074_20160413_dc4ndvi.hdr
lztmre_p104r074_20160413_dc4ndvi.img
lztmre_p104r074_20160413_dc4ndvi.img.aux.xml
lztmre_p104r074_20160429_dc4ndvi.hdr
lztmre_p104r074_20160429_dc4ndvi.img
lztmre_p104r074_20160429_dc4ndvi.img.aux.xml
lztmre_p104r074_20160515_dc4ndvi.hdr
lztmre_p104r074_20160515_dc4ndvi.img
lztmre_p104r074_20160515_dc4ndvi.img.aux.xml
lztmre_p104r074_20160702_dc4ndv

In [53]:
# from pathlib import Path

# tile = "p089r084"  # <-- change if needed
# tile_dir = Path("/home/jovyan/data/compat/files") / tile
# assert tile_dir.exists(), f"Missing: {tile_dir}"
# print("Tile dir:", tile_dir)

In [54]:
import re
from collections import defaultdict


def human_mb(n_bytes: int) -> float:
    return n_bytes / (1024**2)


# regex helpers
re_pair = re.compile(r"_d(\d{8})(\d{8})_")  # e.g. ..._d2023072820240831_...
re_single = re.compile(r"_(\d{8})_")  # e.g. ..._20230728_db8mz...


def img_type(name: str) -> str:
    """
    Returns the product/type token from the end of the filename, e.g.
    db8mz, dc4mz, dllmz, dljmz (and keeps things like vi-fpc_dljmz)
    """
    stem = Path(name).stem  # removes .img
    # take the last underscore chunk as base type
    last = stem.split("_")[-1]
    # special: keep vi-fpc_dljmz or vi-fpc_dllmz together if present
    if "vi-fpc" in stem:
        if stem.endswith("dljmz"):
            return "vi-fpc_dljmz"
        if stem.endswith("dllmz"):
            return "vi-fpc_dllmz"
    return last


def extract_sort_key(p: Path):
    """
    Returns (kind, date1, date2) for sorting:
      - kind: "pair" or "single" or "none"
      - date1/date2: ints YYYYMMDD (or None)
    """
    name = p.name
    m = re_pair.search(name)
    if m:
        d1, d2 = int(m.group(1)), int(m.group(2))
        return ("pair", d1, d2)
    m = re_single.search(name)
    if m:
        d = int(m.group(1))
        return ("single", d, None)
    return ("none", None, None)


# collect
imgs = sorted(tile_dir.rglob("*.img"))

groups = defaultdict(list)
for p in imgs:
    groups[img_type(p.name)].append(p)

# print grouped + sorted
print(f"Found {len(imgs)} .img files\n")

for t in sorted(groups.keys()):
    files = groups[t]
    files_sorted = sorted(files, key=lambda p: extract_sort_key(p))
    print(f"=== {t} ({len(files_sorted)}) ===")
    for p in files_sorted:
        kind, d1, d2 = extract_sort_key(p)
        size = human_mb(p.stat().st_size)
        rel = p.relative_to(tile_dir)
        if kind == "pair":
            print(f"{size:9.2f} MB  {d1}-{d2}  {rel}")
        elif kind == "single":
            print(f"{size:9.2f} MB  {d1}       {rel}")
        else:
            print(f"{size:9.2f} MB            {rel}")
    print()

Found 176 .img files

=== db8mz (2) ===
   624.32 MB  20230503       lztmre_p104r074_20230503_db8mz.img
   624.32 MB  20231026       lztmre_p104r074_20231026_db8mz.img

=== dc4ndvi (171) ===
    52.03 MB  20151223       lztmre_p104r074_20151223_dc4ndvi.img
    52.03 MB  20160209       lztmre_p104r074_20160209_dc4ndvi.img
    52.03 MB  20160328       lztmre_p104r074_20160328_dc4ndvi.img
    52.03 MB  20160413       lztmre_p104r074_20160413_dc4ndvi.img
    52.03 MB  20160429       lztmre_p104r074_20160429_dc4ndvi.img
    52.03 MB  20160515       lztmre_p104r074_20160515_dc4ndvi.img
    52.03 MB  20160702       lztmre_p104r074_20160702_dc4ndvi.img
    52.03 MB  20160718       lztmre_p104r074_20160718_dc4ndvi.img
    52.03 MB  20160803       lztmre_p104r074_20160803_dc4ndvi.img
    52.03 MB  20160819       lztmre_p104r074_20160819_dc4ndvi.img
    52.03 MB  20160904       lztmre_p104r074_20160904_dc4ndvi.img
    52.03 MB  20160920       lztmre_p104r074_20160920_dc4ndvi.img
    52.03 MB  201

In [55]:
# from collections import defaultdict


# def bytes_to_gb(n: int) -> float:
#     return n / (1024**3)


# def bytes_to_mb(n: int) -> float:
#     return n / (1024**2)


# ext_counts = defaultdict(int)
# ext_bytes = defaultdict(int)

# all_files = [p for p in tile_dir.rglob("*") if p.is_file()]

# for p in all_files:
#     ext = p.suffix.lower() if p.suffix else "(no_ext)"
#     ext_counts[ext] += 1
#     ext_bytes[ext] += p.stat().st_size

# # Sort by total bytes desc
# rows = sorted(ext_bytes.keys(), key=lambda e: ext_bytes[e], reverse=True)

# print(f"Total files: {len(all_files)}")
# print(f"Total size:  {bytes_to_gb(sum(ext_bytes.values())):.3f} GB\n")

# print(f"{'EXT':<10} {'COUNT':>8} {'SIZE (MB)':>12} {'SIZE (GB)':>12}")
# print("-" * 46)
# for ext in rows:
#     print(
#         f"{ext:<10} {ext_counts[ext]:>8} {bytes_to_mb(ext_bytes[ext]):>12.2f} {bytes_to_gb(ext_bytes[ext]):>12.3f}"
#     )

# # Quick focused summary:
# for focus in [".img", ".shp"]:
#     print(
#         f"\nFocus {focus}: count={ext_counts[focus]}, size={bytes_to_gb(ext_bytes[focus]):.3f} GB"
#     )

In [56]:
# largest = sorted(all_files, key=lambda p: p.stat().st_size, reverse=True)[:25]
# for p in largest:
#     rel = p.relative_to(tile_dir)
#     size_mb = p.stat().st_size / (1024**2)
#     print(f"{size_mb:9.2f} MB  {rel}")

In [57]:
exclude = set()

# Any file whose *stem* ends with dc4mz is part of the dc4 product family.
# e.g. lztmre_..._20250709_dc4mz.img 
#      lztmre_..._20250709_dc4mz.hdr
#      lztmre_..._20250709_dc4mz.img.aux.xml  (suffix is .xml but name contains .img.aux.xml)
# We'll exclude all files that start with the same base "....dc4mz".
for dc4_img in tile_dir.rglob(f"*dc4{timeseries_data}.img"):
    base = dc4_img.name[:-4]  # remove ".img" -> "....dc4mz"
    for p in dc4_img.parent.iterdir():
        if p.is_file() and p.name.startswith(base):
            exclude.add(p)

print("dc4 products found:", len(list(tile_dir.rglob(f"*dc4{timeseries_data}.img"))))
print("files excluded (dc4 + ancillary):", len(exclude))


dc4 products found: 171
files excluded (dc4 + ancillary): 513


In [58]:
from collections import defaultdict

include = []
for p in tile_dir.rglob("*"):
    if p.is_file() and p not in exclude:
        include.append(p)

def mb(n): return n/(1024**2)
def gb(n): return n/(1024**3)

total_bytes = sum(p.stat().st_size for p in include)
print("Files to zip:", len(include))
print(f"Total size to zip: {gb(total_bytes):.3f} GB")

# quick extension breakdown
ext_counts = defaultdict(int)
ext_bytes = defaultdict(int)
for p in include:
    ext = p.suffix.lower() if p.suffix else "(no_ext)"
    ext_counts[ext] += 1
    ext_bytes[ext] += p.stat().st_size

print("\nEXT           COUNT    SIZE (MB)")
print("-"*30)
for ext in sorted(ext_bytes, key=lambda e: ext_bytes[e], reverse=True):
    print(f"{ext:<10} {ext_counts[ext]:>8} {mb(ext_bytes[ext]):>12.2f}")


Files to zip: 65
Total size to zip: 1.723 GB

EXT           COUNT    SIZE (MB)
------------------------------
.img              5      1560.79
.shp             12       200.54
.dbf             12         2.04
.json             2         0.28
.shx             12         0.28
.xml              5         0.00
.prj             12         0.00
.hdr              5         0.00


In [59]:
# import zipfile

# zip_path = tile_dir.with_name(f"{tile}_{timeseries_data}_no-dc4.zip")
# if zip_path.exists():
#     zip_path.unlink()

# with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
#     for p in include:
#         arcname = p.relative_to(tile_dir.parent)  # keeps "{tile}/..." in zip
#         z.write(p, arcname=str(arcname))

# print("Wrote:", zip_path)
# print("Zip size (GB):", zip_path.stat().st_size / (1024**3))


In [60]:
from pathlib import Path
import zipfile

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def write_zip(tile_dir: Path, tile: str, timeseries_data: str, include: list[Path]) -> Path | None:
    """
    Preview files, ask for confirmation, then write a zip to EXPORT_ROOT.
    Paths are preserved relative to tile_dir.parent.
    """
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_no-dc4.zip"

    # --- Resolve + validate files ---
    files = []
    total_bytes = 0

    existing = []
    missing = []
    skipped = []

    total_bytes = 0
    for p in include:
        p = Path(p)

        # --- EXCLUDES ---
        # Skip any dc4 products
        if "dc4" in p.name.lower():
            skipped.append(p)
            continue

        # (Optional) skip ENVI headers if you only want .img
        # if p.suffix.lower() == ".hdr":
        #     skipped.append(p)
        #     continue

        if p.exists() and p.is_file():
            sz = p.stat().st_size
            existing.append((p, sz))
            total_bytes += sz
        else:
            missing.append(p)


    if not files:
        raise RuntimeError("No valid files to zip")

    # --- Preview ---
    print("\nFiles to be zipped:\n" + "-" * 60)
    for p, size in files:
        rel = p.relative_to(tile_dir.parent)
        print(f"{rel}  ({size / (1024**2):.2f} MB)")

    print("-" * 60)
    print(f"Total files: {len(files)}")
    print(f"Total size:  {total_bytes / (1024**3):.3f} GB")
    print(f"Output ZIP:  {zip_path}\n")

    # --- Confirmation ---
    confirm = input("Proceed with zip? [y/N]: ").strip().lower()
    if confirm != "y":
        print("Aborted — no zip written.")
        return None

    # --- Write ZIP ---
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as z:
        for p, _ in files:
            arcname = p.relative_to(tile_dir.parent)
            z.write(p, arcname=str(arcname))

    print("\nWrote:", zip_path)
    print("Zip size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path


In [61]:
from pathlib import Path
import zipfile
import sys

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def preview_and_zip(tile_dir: Path, tile: str, timeseries_data: str, include, confirm: bool = False) -> Path | None:
    tile_dir = Path(tile_dir)
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_no-dc4.zip"

    print("\n[DEBUG] CWD:", Path.cwd())
    print("[DEBUG] tile_dir:", tile_dir)
    print("[DEBUG] tile_dir.parent:", tile_dir.parent)
    print("[DEBUG] include items:", 0 if include is None else len(include))
    sys.stdout.flush()

    if not include:
        print("[WARN] include is empty / None — nothing to zip.")
        return None

    existing = []
    missing = []

    total_bytes = 0
    existing = []
    missing = []
    skipped = []

    total_bytes = 0
    for p in include:
        print(f"[DEBUG] skipped(dc4) count: {len(skipped)}")
        p = Path(p)

        # --- EXCLUDES ---
        # Skip any dc4 products
        if "dc4" in p.name.lower():
            skipped.append(p)
            continue

        # (Optional) skip ENVI headers if you only want .img
        # if p.suffix.lower() == ".hdr":
        #     skipped.append(p)
        #     continue

        if p.exists() and p.is_file():
            sz = p.stat().st_size
            existing.append((p, sz))
            total_bytes += sz
        else:
            missing.append(p)


    print("\nFiles FOUND (will be zipped):")
    print("-" * 90)
    if existing:
        for p, sz in existing:
            try:
                arcname = p.relative_to(tile_dir.parent)
            except Exception:
                arcname = p.name  # fallback if relative_to fails
            print(f"{arcname}  ({sz/1024**2:.2f} MB)")
    else:
        print("  (none)")
    print("-" * 90)
    print(f"Total files found: {len(existing)}")
    print(f"Total size found:  {total_bytes/1024**3:.3f} GB")
    print(f"Planned ZIP path:  {zip_path}")
    sys.stdout.flush()

    if missing:
        print("\nFiles MISSING (skipped):")
        print("-" * 90)
        for p in missing:
            print(p)
        print("-" * 90)
        sys.stdout.flush()

    if not existing:
        print("[ERR] No existing files to zip — check your 'include' list paths.")
        return None

    print(f"\n[DEBUG] skipped(dc4) count: {len(skipped)}")
    
    if skipped:
        print("\nFiles SKIPPED (excluded - matched 'dc4'):")
        print("-" * 90)
        for p in skipped:
            try:
                rel = p.relative_to(tile_dir.parent)
            except Exception:
                rel = p
            print(rel)
        print("-" * 90)
    else:
        print("\n[OK] No dc4 files were present in 'include' (nothing to exclude).")
    sys.stdout.flush()
    

    if not confirm:
        print("\n[STOP] Not writing ZIP because confirm=False.")
        print("       If the preview looks right, re-run with confirm=True.")
        return None



    # write zip
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p, _ in existing:
            try:
                arcname = p.relative_to(tile_dir.parent)
            except Exception:
                arcname = p.name
            z.write(p, arcname=str(arcname))

    print("\n[OK] Wrote ZIP:", zip_path)
    print("[OK] ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path

# ---- RUN IT (edit these 4 vars to match your notebook) ----
# tile_dir = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074")  # example
# tile = "p104r074"
# timeseries_data = "fc"
# include = [...]  # your list of file Paths

# Preview only:
# preview_and_zip(tile_dir, tile, timeseries_data, include, confirm=False)

# Actually zip:
# preview_and_zip(tile_dir, tile, timeseries_data, include, confirm=True)


In [62]:
from pathlib import Path
import zipfile
import sys

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def preview_and_zip(
    tile_dir: Path,
    tile: str,
    timeseries_data: str,
    include,
    confirm: bool = False,
) -> Path | None:
    """
    Preview + optionally zip a list of files.

    - Excludes anything with 'dc4' in the filename (covers dc4f, dc4mz, etc.)
    - Stores paths inside the zip relative to tile_dir.parent so you get:
        p104r074/<files...> inside the zip
    """
    tile_dir = Path(tile_dir)
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_no-dc4.zip"

    print("\n[DEBUG] CWD:", Path.cwd())
    print("[DEBUG] tile_dir:", tile_dir)
    print("[DEBUG] tile_dir.parent:", tile_dir.parent)
    print("[DEBUG] include items:", 0 if include is None else len(include))
    sys.stdout.flush()

    if not include:
        print("[WARN] include is empty / None — nothing to zip.")
        return None

    existing: list[tuple[Path, int]] = []
    missing: list[Path] = []
    skipped: list[Path] = []
    total_bytes = 0

    # ---- classify files ----
    for p in include:
        p = Path(p)

        # Exclude DC4 products (dc4f, dc4mz, etc.)
        if "dc4" in p.name.lower():
            skipped.append(p)
            continue

        if p.exists() and p.is_file():
            sz = p.stat().st_size
            existing.append((p, sz))
            total_bytes += sz
        else:
            missing.append(p)

    # ---- preview included ----
    print("\nFiles FOUND (will be zipped):")
    print("-" * 90)
    if existing:
        for p, sz in existing:
            try:
                arcname = p.relative_to(tile_dir.parent)
            except Exception:
                arcname = p.name
            print(f"{arcname}  ({sz/1024**2:.2f} MB)")
    else:
        print("  (none)")
    print("-" * 90)
    print(f"Total files found: {len(existing)}")
    print(f"Total size found:  {total_bytes/1024**3:.3f} GB")
    print(f"Planned ZIP path:  {zip_path}")
    sys.stdout.flush()

    # ---- preview missing ----
    if missing:
        print("\nFiles MISSING (skipped because not found):")
        print("-" * 90)
        for p in missing:
            print(p)
        print("-" * 90)
        sys.stdout.flush()

    if not existing:
        print("[ERR] No existing files to zip — check your 'include' list paths.")
        return None

    # ---- preview excluded ----
    print(f"\n[DEBUG] skipped(dc4) count: {len(skipped)}")
    if skipped:
        print("\nFiles SKIPPED (excluded - matched 'dc4'):")
        print("-" * 90)
        for p in skipped:
            try:
                rel = p.relative_to(tile_dir.parent)
            except Exception:
                rel = p
            print(rel)
        print("-" * 90)
    else:
        print("\n[OK] No dc4 files were present in 'include' (nothing to exclude).")
    sys.stdout.flush()

    # ---- confirmation gate ----
    if not confirm:
        print("\n[STOP] Not writing ZIP because confirm=False.")
        print("       If the preview looks right, re-run with confirm=True.")
        return None

    # ---- write zip ----
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as z:
        for p, _ in existing:
            try:
                arcname = p.relative_to(tile_dir.parent)
            except Exception:
                arcname = p.name
            z.write(p, arcname=str(arcname))

    print("\n[OK] Wrote ZIP:", zip_path)
    print("[OK] ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path


# -------------------------
# USAGE (example)
# -------------------------
# tile_dir = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074")
# tile = "p104r074"
# timeseries_data = "fc"
# include = [...]  # list of Paths (or strings) you want to consider
#
# Preview only:
# preview_and_zip(tile_dir, tile, timeseries_data, include, confirm=False)
#
# Actually write:
# preview_and_zip(tile_dir, tile, timeseries_data, include, confirm=True)


In [63]:
from pathlib import Path
import zipfile
import sys
import random

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def preview_and_zip(
    tile_dir: Path,
    tile: str,
    timeseries_data: str,
    confirm: bool = False,
    n_dc4: int = 4,
) -> Path | None:
    tile_dir = Path(tile_dir)
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_with-{n_dc4}-dc4.zip"

    dc4_tag = f"dc4{timeseries_data.lower()}"  # e.g. dc4fc

    print("\n[DEBUG] CWD:", Path.cwd())
    print("[DEBUG] tile_dir:", tile_dir)
    print("[DEBUG] dc4 tag:", dc4_tag)
    sys.stdout.flush()

    # -------------------------------------------------
    # Scan files from disk
    # -------------------------------------------------
    all_files = sorted([p for p in tile_dir.rglob("*") if p.is_file()])

    dc4_files = [p for p in all_files if dc4_tag in p.name.lower()]
    non_dc4_files = [p for p in all_files if dc4_tag not in p.name.lower()]

    # -------------------------------------------------
    # Group dc4 files by product (date-based stem)
    # -------------------------------------------------
    dc4_groups: dict[str, list[Path]] = {}
    for p in dc4_files:
        # strip extensions: .img, .hdr, .img.aux.xml
        stem = p.name.replace(".img.aux.xml", "").replace(".img", "").replace(".hdr", "")
        dc4_groups.setdefault(stem, []).append(p)

    dc4_keys = sorted(dc4_groups.keys())

    if not dc4_keys:
        print("[WARN] No DC4 files found on disk")
        selected_dc4 = {}
    else:
        sample_n = min(n_dc4, len(dc4_keys))
        selected_keys = random.sample(dc4_keys, sample_n)
        selected_dc4 = {k: dc4_groups[k] for k in selected_keys}

    # -------------------------------------------------
    # Build include list
    # -------------------------------------------------
    include = []
    include.extend(non_dc4_files)

    for files in selected_dc4.values():
        include.extend(files)

    # -------------------------------------------------
    # Preview
    # -------------------------------------------------
    print(f"\nDC4 products found: {len(dc4_keys)}")
    print(f"DC4 products selected: {len(selected_dc4)}")
    print("-" * 90)

    for k in selected_dc4:
        print("INCLUDE DC4:", k)
    print("-" * 90)

    excluded_dc4 = [k for k in dc4_keys if k not in selected_dc4]
    print(f"DC4 products excluded: {len(excluded_dc4)}")
    if excluded_dc4:
        for k in excluded_dc4[:10]:
            print("EXCLUDE DC4:", k)
        if len(excluded_dc4) > 10:
            print(f"... plus {len(excluded_dc4) - 10} more")
    sys.stdout.flush()

    # -------------------------------------------------
    # Size + file preview
    # -------------------------------------------------
    total_bytes = sum(p.stat().st_size for p in include)
    print("\nTotal files to zip:", len(include))
    print("Total size (GB):", round(total_bytes / (1024**3), 3))
    print("Planned ZIP:", zip_path)
    sys.stdout.flush()

    if not confirm:
        print("\n[STOP] Preview only — re-run with confirm=True to write ZIP.")
        return None

    # -------------------------------------------------
    # Write ZIP
    # -------------------------------------------------
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as z:
        for p in include:
            try:
                arcname = p.relative_to(tile_dir.parent)
            except Exception:
                arcname = p.name
            z.write(p, arcname=str(arcname))

    print("\n[OK] Wrote ZIP:", zip_path)
    print("[OK] ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path


# -------------------------------------------------
# USAGE
# -------------------------------------------------
# tile_dir = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074")
# tile = "p104r074"
# timeseries_data = "fc"

# Preview (no zip)
preview_and_zip(tile_dir, tile, timeseries_data, confirm=False, n_dc4=4)

# Write zip
# preview_and_zip(tile_dir, tile, timeseries_data, confirm=True, n_dc4=4)



[DEBUG] CWD: /home/jovyan/work-easi-eds/notebooks/zip_contents
[DEBUG] tile_dir: /home/jovyan/work-easi-eds/data/compat/files/ndvi/p104r074
[DEBUG] dc4 tag: dc4ndvi

DC4 products found: 171
DC4 products selected: 4
------------------------------------------------------------------------------------------
INCLUDE DC4: lztmre_p104r074_20190625_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20220305_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20190727_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20191202_dc4ndvi
------------------------------------------------------------------------------------------
DC4 products excluded: 167
EXCLUDE DC4: lztmre_p104r074_20151223_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160209_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160328_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160413_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160429_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160515_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160702_dc4ndvi
EXCLUDE DC4: lztmre_p104r074_20160718_dc4ndvi
EXCLUDE DC4: lztmre_p10

In [64]:
include_core =include

In [65]:
from pathlib import Path
import zipfile
import sys
import random

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def preview_and_zip_with_random_dc4(
    tile_dir: Path,
    tile: str,
    timeseries_data: str,
    include_core,                 # your curated list (e.g. the 25 files)
    n_dc4: int = 4,               # how many dc4 products to include
    seed: int | None = None,      # set to an int for reproducible random
    confirm: bool = False
) -> Path | None:
    tile_dir = Path(tile_dir)
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_core_plus_{n_dc4}_dc4.zip"

    dc4_tag = f"dc4{timeseries_data.lower()}"  # e.g. "dc4fc"

    print("\n[DEBUG] CWD:", Path.cwd())
    print("[DEBUG] tile_dir:", tile_dir)
    print("[DEBUG] dc4_tag:", dc4_tag)
    print("[DEBUG] core include items:", 0 if include_core is None else len(include_core))
    sys.stdout.flush()

    if not include_core:
        print("[WARN] include_core is empty / None — nothing to zip.")
        return None

    # -----------------------------
    # Validate / normalise core list
    # -----------------------------
    core_existing = []
    core_missing = []
    for p in include_core:
        p = Path(p)
        if p.exists() and p.is_file():
            core_existing.append(p)
        else:
            core_missing.append(p)

    if core_missing:
        print("\n[WARN] Core files missing (will be skipped):")
        for p in core_missing[:25]:
            print(" -", p)
        if len(core_missing) > 25:
            print(f"... plus {len(core_missing) - 25} more")
        sys.stdout.flush()

    if not core_existing:
        print("[ERR] No core files exist on disk.")
        return None

    # -----------------------------
    # Find all dc4 files ON DISK and group them
    # -----------------------------
    all_dc4_files = [p for p in tile_dir.iterdir() if p.is_file() and dc4_tag in p.name.lower()]

    dc4_groups: dict[str, list[Path]] = {}
    for p in all_dc4_files:
        stem = p.name.replace(".img.aux.xml", "").replace(".img", "").replace(".hdr", "")
        dc4_groups.setdefault(stem, []).append(p)

    dc4_keys = sorted(dc4_groups.keys())
    print(f"\n[DEBUG] dc4 products available on disk: {len(dc4_keys)}")
    sys.stdout.flush()

    selected_dc4_files = []
    selected_keys = []

    if dc4_keys:
        if seed is not None:
            random.seed(seed)

        sample_n = min(n_dc4, len(dc4_keys))
        selected_keys = random.sample(dc4_keys, sample_n)

        for k in selected_keys:
            # add all ancillary files for the selected product
            selected_dc4_files.extend(dc4_groups[k])

    # -----------------------------
    # Build final include list
    #   - core outputs
    #   - + selected dc4 groups
    # -----------------------------
    include_final = list(core_existing)
    include_final.extend(selected_dc4_files)

    # De-dup (just in case)
    include_final = sorted(set(include_final))

    # -----------------------------
    # Preview
    # -----------------------------
    print("\nSelected DC4 products to INCLUDE:")
    print("-" * 90)
    if selected_keys:
        for k in selected_keys:
            print("INCLUDE DC4:", k)
    else:
        print("(none selected / none available)")
    print("-" * 90)

    # Show core files (like your old output)
    print("\nCore files FOUND (will be zipped):")
    print("-" * 90)
    total_bytes = 0
    for p in core_existing:
        rel = p.relative_to(tile_dir.parent)
        size = p.stat().st_size
        total_bytes += size
        print(f"{rel}  ({size/1024**2:.2f} MB)")
    print("-" * 90)

    # Show dc4 files being added
    print("\nDC4 files ADDED (with ancillary files):")
    print("-" * 90)
    if selected_dc4_files:
        for p in sorted(selected_dc4_files):
            rel = p.relative_to(tile_dir.parent)
            size = p.stat().st_size
            total_bytes += size
            print(f"{rel}  ({size/1024**2:.2f} MB)")
    else:
        print("(none)")
    print("-" * 90)

    print(f"Total files to zip: {len(include_final)}")
    print(f"Total size (GB):    {total_bytes/1024**3:.3f}")
    print(f"Planned ZIP path:   {zip_path}")
    sys.stdout.flush()

    if not confirm:
        print("\n[STOP] Preview only — re-run with confirm=True to write ZIP.")
        return None

    # -----------------------------
    # Write ZIP
    # -----------------------------
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in include_final:
            arcname = p.relative_to(tile_dir.parent)  # keeps "p104r074/..." inside zip
            z.write(p, arcname=str(arcname))

    print("\n[OK] Wrote ZIP:", zip_path)
    print("[OK] ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path


# -----------------------------
# USAGE
# -----------------------------
# tile_dir = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074")
# tile = "p104r074"
# timeseries_data = "fc"

# Your existing curated list goes here (the 25 files you already build)
# include_core = [...]

# Preview:
# preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=False)

# Write:
# preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=True)


In [66]:
preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=False)



[DEBUG] CWD: /home/jovyan/work-easi-eds/notebooks/zip_contents
[DEBUG] tile_dir: /home/jovyan/work-easi-eds/data/compat/files/ndvi/p104r074
[DEBUG] dc4_tag: dc4ndvi
[DEBUG] core include items: 65

[DEBUG] dc4 products available on disk: 171

Selected DC4 products to INCLUDE:
------------------------------------------------------------------------------------------
INCLUDE DC4: lztmre_p104r074_20220422_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20200323_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20200627_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20221108_dc4ndvi
------------------------------------------------------------------------------------------

Core files FOUND (will be zipped):
------------------------------------------------------------------------------------------
p104r074/lztmre_p104r074_20231026_db8mz.img.aux.xml  (0.00 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-ndvi_dljmz.img  (208.11 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-ndvi_dllmz_log.json  (0.00 MB)
p104r074/lztmna_

In [67]:
preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=True)


[DEBUG] CWD: /home/jovyan/work-easi-eds/notebooks/zip_contents
[DEBUG] tile_dir: /home/jovyan/work-easi-eds/data/compat/files/ndvi/p104r074
[DEBUG] dc4_tag: dc4ndvi
[DEBUG] core include items: 65

[DEBUG] dc4 products available on disk: 171

Selected DC4 products to INCLUDE:
------------------------------------------------------------------------------------------
INCLUDE DC4: lztmre_p104r074_20200323_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20201001_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20221015_dc4ndvi
INCLUDE DC4: lztmre_p104r074_20230831_dc4ndvi
------------------------------------------------------------------------------------------

Core files FOUND (will be zipped):
------------------------------------------------------------------------------------------
p104r074/lztmre_p104r074_20231026_db8mz.img.aux.xml  (0.00 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-ndvi_dljmz.img  (208.11 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-ndvi_dllmz_log.json  (0.00 MB)
p104r074/lztmna_

PosixPath('/home/jovyan/work-easi-eds/exports/zips/p104r074_ndvi_core_plus_4_dc4.zip')

In [38]:
from pathlib import Path
import zipfile
import sys
import random

EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def preview_and_zip_with_random_dc4(
    tile_dir: Path,
    tile: str,
    timeseries_data: str,
    include_core,                 # your curated list (e.g. the 25 files)
    n_dc4: int = 4,               # how many dc4 products to include
    seed: int | None = None,      # set to an int for reproducible random
    confirm: bool = False
) -> Path | None:
    tile_dir = Path(tile_dir)
    zip_path = EXPORT_ROOT / f"{tile}_{timeseries_data}_core_plus_{n_dc4}_dc4.zip"

    dc4_tag = f"dc4{timeseries_data.lower()}"  # e.g. "dc4fc"

    print("\n[DEBUG] CWD:", Path.cwd())
    print("[DEBUG] tile_dir:", tile_dir)
    print("[DEBUG] dc4_tag:", dc4_tag)
    print("[DEBUG] core include items:", 0 if include_core is None else len(include_core))
    sys.stdout.flush()

    if not include_core:
        print("[WARN] include_core is empty / None — nothing to zip.")
        return None

    # -----------------------------
    # Validate / normalise core list
    # -----------------------------
    core_existing = []
    core_missing = []
    for p in include_core:
        p = Path(p)
        if p.exists() and p.is_file():
            core_existing.append(p)
        else:
            core_missing.append(p)

    if core_missing:
        print("\n[WARN] Core files missing (will be skipped):")
        for p in core_missing[:25]:
            print(" -", p)
        if len(core_missing) > 25:
            print(f"... plus {len(core_missing) - 25} more")
        sys.stdout.flush()

    if not core_existing:
        print("[ERR] No core files exist on disk.")
        return None

    # -----------------------------
    # Find all dc4 files ON DISK and group them
    # -----------------------------
    all_dc4_files = [p for p in tile_dir.iterdir() if p.is_file() and dc4_tag in p.name.lower()]

    dc4_groups: dict[str, list[Path]] = {}
    for p in all_dc4_files:
        stem = p.name.replace(".img.aux.xml", "").replace(".img", "").replace(".hdr", "")
        dc4_groups.setdefault(stem, []).append(p)

    dc4_keys = sorted(dc4_groups.keys())
    print(f"\n[DEBUG] dc4 products available on disk: {len(dc4_keys)}")
    sys.stdout.flush()

    selected_dc4_files = []
    selected_keys = []

    if dc4_keys:
        if seed is not None:
            random.seed(seed)

        sample_n = min(n_dc4, len(dc4_keys))
        selected_keys = random.sample(dc4_keys, sample_n)

        for k in selected_keys:
            # add all ancillary files for the selected product
            selected_dc4_files.extend(dc4_groups[k])

    # -----------------------------
    # Build final include list
    #   - core outputs
    #   - + selected dc4 groups
    # -----------------------------
    include_final = list(core_existing)
    include_final.extend(selected_dc4_files)

    # De-dup (just in case)
    include_final = sorted(set(include_final))

    # -----------------------------
    # Preview
    # -----------------------------
    print("\nSelected DC4 products to INCLUDE:")
    print("-" * 90)
    if selected_keys:
        for k in selected_keys:
            print("INCLUDE DC4:", k)
    else:
        print("(none selected / none available)")
    print("-" * 90)

    # Show core files (like your old output)
    print("\nCore files FOUND (will be zipped):")
    print("-" * 90)
    total_bytes = 0
    for p in core_existing:
        rel = p.relative_to(tile_dir.parent)
        size = p.stat().st_size
        total_bytes += size
        print(f"{rel}  ({size/1024**2:.2f} MB)")
    print("-" * 90)

    # Show dc4 files being added
    print("\nDC4 files ADDED (with ancillary files):")
    print("-" * 90)
    if selected_dc4_files:
        for p in sorted(selected_dc4_files):
            rel = p.relative_to(tile_dir.parent)
            size = p.stat().st_size
            total_bytes += size
            print(f"{rel}  ({size/1024**2:.2f} MB)")
    else:
        print("(none)")
    print("-" * 90)

    print(f"Total files to zip: {len(include_final)}")
    print(f"Total size (GB):    {total_bytes/1024**3:.3f}")
    print(f"Planned ZIP path:   {zip_path}")
    sys.stdout.flush()

    if not confirm:
        print("\n[STOP] Preview only — re-run with confirm=True to write ZIP.")
        return None

    # -----------------------------
    # Write ZIP
    # -----------------------------
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in include_final:
            arcname = p.relative_to(tile_dir.parent)  # keeps "p104r074/..." inside zip
            z.write(p, arcname=str(arcname))

    print("\n[OK] Wrote ZIP:", zip_path)
    print("[OK] ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    return zip_path


# -----------------------------
# USAGE
# -----------------------------
# tile_dir = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074")
# tile = "p104r074"
# timeseries_data = "fc"
# include_core = include
# Your existing curated list goes here (the 25 files you already build)
# include_core = [...]

# Preview:
# preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=False)

# Write:
# preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=True)


In [39]:
preview_and_zip_with_random_dc4(tile_dir, tile, timeseries_data, include_core, n_dc4=4, seed=None, confirm=False)


[DEBUG] CWD: /home/jovyan/work-easi-eds/notebooks/zip_contents
[DEBUG] tile_dir: /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074
[DEBUG] dc4_tag: dc4fc
[DEBUG] core include items: 17

[DEBUG] dc4 products available on disk: 243

Selected DC4 products to INCLUDE:
------------------------------------------------------------------------------------------
INCLUDE DC4: lztmre_p104r074_20240801_dc4fc
INCLUDE DC4: lztmre_p104r074_20180622_dc4fc
INCLUDE DC4: lztmre_p104r074_20231127_dc4fc
INCLUDE DC4: lztmre_p104r074_20171025_dc4fc
------------------------------------------------------------------------------------------

Core files FOUND (will be zipped):
------------------------------------------------------------------------------------------
p104r074/lztmre_p104r074_d2023050320231026_vi-fc_dljmz.img  (208.11 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-fc_dllmz_log.json  (0.00 MB)
p104r074/lztmre_p104r074_20231026_db8mz.img.aux.xml  (0.00 MB)
p104r074/lztmna_p104r074_eall_dw

In [40]:
preview_and_zip_with_random_dc4(
    tile_dir,
    tile,
    timeseries_data,
    include_core,
    n_dc4=4,
    seed=None,
    confirm=True,
)



[DEBUG] CWD: /home/jovyan/work-easi-eds/notebooks/zip_contents
[DEBUG] tile_dir: /home/jovyan/work-easi-eds/data/compat/files/fc/p104r074
[DEBUG] dc4_tag: dc4fc
[DEBUG] core include items: 17

[DEBUG] dc4 products available on disk: 243

Selected DC4 products to INCLUDE:
------------------------------------------------------------------------------------------
INCLUDE DC4: lztmre_p104r074_20230519_dc4fc
INCLUDE DC4: lztmre_p104r074_20220217_dc4fc
INCLUDE DC4: lztmre_p104r074_20250305_dc4fc
INCLUDE DC4: lztmre_p104r074_20180214_dc4fc
------------------------------------------------------------------------------------------

Core files FOUND (will be zipped):
------------------------------------------------------------------------------------------
p104r074/lztmre_p104r074_d2023050320231026_vi-fc_dljmz.img  (208.11 MB)
p104r074/lztmre_p104r074_d2023050320231026_vi-fc_dllmz_log.json  (0.00 MB)
p104r074/lztmre_p104r074_20231026_db8mz.img.aux.xml  (0.00 MB)
p104r074/lztmna_p104r074_eall_dw

PosixPath('/home/jovyan/work-easi-eds/exports/zips/p104r074_fc_core_plus_4_dc4.zip')